# Problem Book P1, Chapter 5: Finite Fields at Play (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy`, `sympy`, `scipy` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import itertools, math, random
from math import comb
from fractions import Fraction as Fr
from collections import Counter
import numpy as np
import sympy as sp
from sympy import legendre_symbol, primerange, divisors, mobius, isprime
import matplotlib.pyplot as plt
random.seed(5)

def chi(a, p):                       # Legendre symbol with chi(0) = 0
    a %= p
    return 0 if a == 0 else legendre_symbol(a, p)
def M(n, q):                         # monic irreducibles of degree n over F_q (Tome 2, Theorem 9.10)
    return sum(mobius(d) * q**(n // d) for d in divisors(n)) // n

## Part A: checks

**Problem 5.1 (conics).** The character sum of a quadratic, and the number of points of $Q(x,y)=1$, for every form with
non-zero discriminant and $p\le13$.

In [ ]:
ok = True
for p in (5, 7, 11, 13):
    for a in range(1, p):
        for b in range(p):
            for c in range(p):
                D = (b * b - 4 * a * c) % p
                s = sum(chi(a * x * x + b * x + c, p) for x in range(p))
                ok &= s == (-chi(a, p) if D else (p - 1) * chi(a, p))
                if D and c:
                    N = sum(1 for x in range(p) for y in range(p) if (a * x * x + b * x * y + c * y * y) % p == 1)
                    ok &= N == p - chi(D, p)
print("conics:", ok)

**Problem 5.2 (sums of squares).** The formula for $N_n(c)$, and the total variation distance to uniform.

In [ ]:
ok = True
for p in (3, 5, 7):
    for n in range(1, 5):
        cnt = Counter(sum(t * t for t in x) % p for x in itertools.product(range(p), repeat=n))
        for c in range(p):
            if n % 2 == 0:
                pred = p**(n - 1) + p**(n // 2 - 1) * chi((-1)**(n // 2), p) * ((p - 1) if c == 0 else -1)
            else:
                pred = p**(n - 1) + (p**((n - 1) // 2) * chi((-1)**((n - 1) // 2) * c, p) if c else 0)
            ok &= cnt[c] == pred
        tv = Fr(sum(abs(Fr(cnt[c], p**n) - Fr(1, p)) for c in range(p)), 2)
        ok &= tv == (Fr(p - 1, p) / p**Fr(n // 2) if n % 2 == 0 else Fr(p - 1, 2) / p**((n + 1) // 2))
print("sums of squares:", ok)

**Problem 5.3 (Gauss's cubic).** $N=p-2+a$ with $a\equiv1\pmod 3$ and $4p=a^2+27b^2$, for all $p\equiv1\pmod3$ below $1000$.

In [ ]:
def cubic_count(p):
    cubes = Counter(pow(x, 3, p) for x in range(p))
    return sum(c * cubes[(1 - u) % p] for u, c in cubes.items())
ok, rows = True, []
for p in primerange(7, 1000):
    if p % 3 != 1: continue
    a = cubic_count(p) - p + 2
    b2 = Fr(4 * p - a * a, 27)
    ok &= a % 3 == 1 and b2.denominator == 1 and sp.sqrt(b2).is_integer and abs(a) <= 2 * math.sqrt(p)
    rows.append((p, cubic_count(p), a))
print(ok, rows[:4])

**Problem 5.4 (nilpotent matrices).** Brute force, and the identity $\sum_k q^{\binom k2}/D_k=q^{\binom{n+1}2}/D_n$.

In [ ]:
def nilpotent_count(n, q):
    cnt = 0
    for e in itertools.product(range(q), repeat=n * n):
        A = np.array(e).reshape(n, n); P = A.copy()
        for _ in range(n - 1): P = P @ A % q
        cnt += not P.any()
    return cnt
print([(n, q, nilpotent_count(n, q), q**(n * n - n)) for n, q in ((2, 2), (2, 3), (3, 2), (2, 5))])
D = lambda k, q: math.prod(q**i - 1 for i in range(1, k + 1))
print(all(sum(Fr(q**comb(k, 2), D(k, q)) for k in range(n + 1)) == Fr(q**comb(n + 1, 2), D(n, q))
          for q in (2, 3, 4, 5) for n in range(8)))

**Problem 5.5 (rank).** Counts by rank add up to $q^{n^2}$; the limit law for $q=2$; Durfee's identity as power series.

In [ ]:
def rank_count(n, r, q):
    num = den = 1
    for i in range(r): num *= (q**n - q**i)**2; den *= q**r - q**i
    return num // den
print(all(sum(rank_count(n, r, q) for r in range(n + 1)) == q**(n * n) for q in (2, 3, 5) for n in range(1, 7)))
q, n = 2, 60
lim = lambda k: q**(-k * k) * math.prod(1 - q**-j for j in range(k + 1, 300)) / math.prod(1 - q**-j for j in range(1, k + 1))
print([round(rank_count(n, n - k, q) / q**(n * n), 6) for k in range(4)], [round(lim(k), 6) for k in range(4)])
N = 40                                           # power series in x, as integer coefficient lists mod x^N
def mul(a, b): return [sum(a[i] * b[k - i] for i in range(k + 1)) for k in range(N)]
def inv_prod(J, power):                          # 1 / prod_{j<=J} (1 - x^j)^power
    c = [1] + [0] * (N - 1)
    for j in range(1, J + 1):
        for _ in range(power):
            for k in range(j, N): c[k] += c[k - j]
    return c
lhs = [0] * N
for k in range(7):
    term = inv_prod(k, 2)
    for i in range(N - k * k): lhs[i + k * k] += term[i]
print("Durfee up to x^39:", lhs == inv_prod(N, 1))

**Problem 5.6 (flags).** Complements of a line in $\mathbb F_2^3$ and $\mathbb F_3^3$ by brute force, and the order of $GL_n$.

In [ ]:
def subspaces(n, q, k):                          # all k-dimensional subspaces of F_q^n, as frozensets
    vecs = list(itertools.product(range(q), repeat=n)); out = set()
    for B in itertools.combinations(vecs[1:], k):
        span = frozenset(tuple(sum(c * b[i] for c, b in zip(cs, B)) % q for i in range(n))
                         for cs in itertools.product(range(q), repeat=k))
        if len(span) == q**k: out.add(span)
    return out
for q in (2, 3):
    U = frozenset(tuple(t * u % q for u in (1, 0, 0)) for t in range(q))
    comps = [W for W in subspaces(3, q, 2) if len(W & U) == 1]
    print(q, len(comps), q**(1 * 2))
GL = lambda n, q: math.prod(q**n - q**i for i in range(n))
qfact = lambda n, q: math.prod((q**i - 1) // (q - 1) for i in range(1, n + 1))
print(all(GL(n, q) == q**comb(n, 2) * (q - 1)**n * qfact(n, q) for q in (2, 3, 4, 5, 7) for n in range(1, 7)))

**Problem 5.7 (Edmonds).** The random determinant test against a brute-force matching search, on random bipartite graphs.

In [ ]:
def has_matching(adj, n):
    return any(all(adj[i][s[i]] for i in range(n)) for s in itertools.permutations(range(n)))
def edmonds_test(adj, n, p=10007):
    X = sp.Matrix(n, n, lambda i, j: random.randrange(p) if adj[i][j] else 0)
    return X.det() % p != 0
agree, wrong = 0, 0
for _ in range(200):
    n = 5; adj = [[random.random() < 0.35 for _ in range(n)] for _ in range(n)]
    h, t = has_matching(adj, n), edmonds_test(adj, n)
    agree += h == t; wrong += (t and not h)
print(agree, "agreements out of 200; false 'matching' answers:", wrong)

**Problem 5.8 (permanent lemma).** For random $3\times3$ matrices over $\mathbb F_5$ with non-zero permanent, random $b$ and random
pairs $S_i$, a good $x$ always exists.

In [ ]:
p, ok, tried = 5, True, 0
while tried < 300:
    A = [[random.randrange(p) for _ in range(3)] for _ in range(3)]
    per = sum(math.prod(A[i][s[i]] for i in range(3)) for s in itertools.permutations(range(3))) % p
    if per == 0: continue
    tried += 1
    b = [random.randrange(p) for _ in range(3)]; S = [random.sample(range(p), 2) for _ in range(3)]
    ok &= any(all(sum(A[i][j] * x[j] for j in range(3)) % p != b[i] for i in range(3)) for x in itertools.product(*S))
print("permanent lemma:", ok)

**Problems 5.9 and 5.10 (Cauchy--Davenport, Erdős--Heilbronn).** Exhaustive for small $p$; and the bound for subset sums.

In [ ]:
ok_cd = ok_eh = True
for p in (5, 7, 11):
    for k in range(1, p + 1):
        for A in itertools.combinations(range(p), k):
            ok_eh &= len({(a + b) % p for a in A for b in A if a != b}) >= min(p, 2 * k - 3)
            if k <= 3 and p <= 7:
                for l in range(1, 4):
                    for B in itertools.combinations(range(p), l):
                        ok_cd &= len({(a + b) % p for a in A for b in B}) >= min(p, k + l - 1)
print("Cauchy-Davenport:", ok_cd, " Erdos-Heilbronn:", ok_eh)
p = 11
for _ in range(200):
    a = [random.randrange(1, p) for _ in range(p - 1)]
    sums = {0}
    for t in a: sums |= {(s + t) % p for s in sums}
    assert len(sums) == p
print("p-1 non-zero residues give all subset sums; p-2 ones miss", p - 1)
print("Catalan:", [comb(2 * k - 4, k - 2) - comb(2 * k - 4, k - 1) for k in range(2, 9)])

**Problem 5.11 (Erdős--Ginzburg--Ziv).** All multisets of $2n-1$ residues for $n\le5$, and the sharpness example.

In [ ]:
def egz(n):
    return all(any(sum(c) % n == 0 for c in itertools.combinations(seq, n))
               for seq in itertools.combinations_with_replacement(range(n), 2 * n - 1))
print([egz(n) for n in range(1, 6)],
      [any(sum(c) % n == 0 for c in itertools.combinations([0] * (n - 1) + [1] * (n - 1), n)) for n in range(2, 6)])

**Problem 5.12 (hyperplanes).** In $\mathbb F_3^2$: the fewest lines covering everything but $0$, and the smallest blocking set.

In [ ]:
q = 3
pts = [(x, y) for x in range(q) for y in range(q)]
lines = {frozenset(((a[0] + t * d[0]) % q, (a[1] + t * d[1]) % q) for t in range(q))
         for a in pts for d in ((1, 0), (0, 1), (1, 1), (1, 2))}
avoid = [L for L in lines if (0, 0) not in L]
cover = next(m for m in range(1, 9) if any(set().union(*Ls) == set(pts) - {(0, 0)} for Ls in itertools.combinations(avoid, m)))
block = next(m for m in range(1, 9) if any(all(L & set(B) for L in lines) for B in itertools.combinations(pts, m)))
print("cover:", cover, "=", 2 * (q - 1), "  blocking:", block, "=", 2 * (q - 1) + 1)

**Problem 5.13 (Kakeya).** The minimum in $\mathbb F_3^2$ by brute force, and the construction of (b) for $q=3,5,7,11$.

In [ ]:
def is_kakeya(K, q):
    dirs = [(1, m) for m in range(q)] + [(0, 1)]
    return all(any(all(((a + t * d[0]) % q, (b + t * d[1]) % q) in K for t in range(q)) for a in range(q) for b in range(q))
               for d in dirs)
q = 3
P = [(x, y) for x in range(q) for y in range(q)]
print("min Kakeya in F_3^2:", next(k for k in range(1, 10) if any(is_kakeya(set(K), q) for K in itertools.combinations(P, k))),
      " Dvir bound:", comb(q + 1, 2))
for q in (3, 5, 7, 11):
    K = {(x, (m * x + m * m) % q) for m in range(q) for x in range(q)} | {(0, y) for y in range(q)}
    print(q, is_kakeya(K, q), len(K), q * (q + 1) // 2 + (q - 1) // 2)

**Problem 5.14 (Moore).** The determinant identity for $(q,n)=(2,3)$ and $(3,2)$.

In [ ]:
for q, n in ((2, 3), (3, 2)):
    xs = sp.symbols(f'x1:{n + 1}')
    Mo = sp.Matrix(n, n, lambda i, j: xs[i]**(q**j))
    rhs = 1
    for i in range(n):
        for cs in itertools.product(range(q), repeat=i):
            rhs *= xs[i] + sum(c * xs[j] for j, c in enumerate(cs))
    print(q, n, sp.Poly(Mo.det(), *xs, modulus=q) == sp.Poly(sp.expand(rhs), *xs, modulus=q))

**Problem 5.15 (cap sets).** The bound $3M_n$, the constant $\theta$, and a check that the cap of Figure 5.2 is a cap.

In [ ]:
def Mn(n):
    c = [1]
    for _ in range(n): c = [sum(c[i - j] for j in range(3) if 0 <= i - j < len(c)) for i in range(len(c) + 2)]
    return sum(c[:2 * n // 3 + 1])
print([(n, 3 * Mn(n), 3**n) for n in range(1, 9)])
t = (math.sqrt(33) - 1) / 8; print("t =", t, " theta =", (1 + t + t * t) / t**(2 / 3))
print("M_n^(1/n):", [round(math.exp(math.log(Mn(n)) / n), 4) for n in (30, 300, 1500)])

**Problem 5.16 (Paley).** Parameters, triangles, clique numbers, and the discrepancy ratio on random sets.

In [ ]:
def paley_clique(p):              # largest clique through 0, searched among the squares
    sq = {t * t % p for t in range(1, p)}
    best = [1]
    def grow(K, cand):
        best[0] = max(best[0], len(K))
        for i, v in enumerate(cand):
            if len(K) + len(cand) - i <= best[0]: return
            grow(K + [v], [w for w in cand[i + 1:] if (w - v) % p in sq])
    grow([0], sorted(sq)); return best[0]
for p in (13, 17, 29, 37):
    sq = {t * t % p for t in range(1, p)}
    lam = {sum(1 for z in range(p) if (z in sq) and ((z - d) % p in sq)) for d in sq}
    mu = {sum(1 for z in range(p) if (z in sq) and ((z - d) % p in sq)) for d in range(1, p) if d not in sq}
    tri = sum(1 for a, b, c in itertools.combinations(range(p), 3) if (b - a) % p in sq and (c - b) % p in sq and (c - a) % p in sq)
    print(p, lam, (p - 5) // 4, mu, (p - 1) // 4, tri, p * (p - 1) * (p - 5) // 48, "clique", paley_clique(p), round(math.sqrt(p), 2))
p = 101; sq = {t * t % p for t in range(1, p)}; worst = 0
for _ in range(1500):
    S = random.sample(range(p), random.randint(1, p)); T = random.sample(range(p), random.randint(1, p))
    e = sum(1 for s in S for t in T if (s - t) % p in sq)
    worst = max(worst, abs(e - (len(S) * len(T) - len(set(S) & set(T))) / 2) / math.sqrt(p * len(S) * len(T)))
print("largest discrepancy ratio seen:", round(worst, 4), "(bound 0.5)")

**Problem 5.17 (m-sequences).** Period, windows, balance and autocorrelation for $x^4+x+1$ and $x^5+x^2+1$.

In [ ]:
def lfsr(taps, n, N, start=None):          # s_{k+n} = sum of s_{k+t} for t in taps (mod 2)
    s = list(start or [0] * (n - 1) + [1])
    while len(s) < N: s.append(sum(s[-n + t] for t in taps) % 2)
    return s
for taps, n in (([0, 1], 4), ([0, 2], 5)):
    L = 2**n - 1; s = lfsr(taps, n, 3 * L)
    period = next(T for T in range(1, L + 1) if s[T:T + 2 * n] == s[:2 * n])
    windows = {tuple(s[k:k + n]) for k in range(L)}
    ac = {sum((-1)**(s[k] + s[k + t]) for k in range(L)) for t in range(1, L)}
    print(n, "period", period, "windows", len(windows), "ones", sum(s[:L]), "autocorrelations", ac)
print("".join(map(str, lfsr([0, 1], 4, 15, [0, 0, 0, 1]))))

**Problem 5.18 (Singer).** The three examples, and Singer sets rebuilt from $\mathbb F_{q^3}=\mathbb F_q[x]/(f)$ for $q=2,3,5,7$.

In [ ]:
def perfect(D, N): return sorted((a - b) % N for a in D for b in D if a != b) == list(range(1, N))
print(perfect([0, 1, 3], 7), perfect([0, 1, 3, 9], 13), perfect([0, 1, 4, 14, 16], 21))
def singer(q):
    x = sp.symbols('x'); N3 = q**3 - 1
    for c in itertools.product(range(q), repeat=3):
        f = sp.Poly([1, *c], x, modulus=q)
        if f.is_irreducible and all(sp.rem(sp.Poly(x, x, modulus=q)**(N3 // r), f) != sp.Poly(1, x, modulus=q)
                                    for r in sp.primefactors(N3)): break
    Nn, D, g = q * q + q + 1, [], sp.Poly(1, x, modulus=q)
    for i in range(Nn):                          # U = {a + b x}: the coefficient of x^2 vanishes
        co = g.all_coeffs(); co = [0] * (3 - len(co)) + co
        if co[0] % q == 0: D.append(i)
        g = sp.rem(g * sp.Poly(x, x, modulus=q), f)
    return D, Nn
for q in (2, 3, 5, 7):
    D, Nn = singer(q); print(q, D, Nn, perfect(D, Nn))

**Problem 5.19 (Sidon).** The Erdős--Turán construction, and exact values of $F(n)$ for small $n$.

In [ ]:
for p in (5, 7, 11, 13):
    A = [2 * p * k + (k * k) % p for k in range(p)]
    sums = [a + b for i, a in enumerate(A) for b in A[i:]]
    print(p, len(sums) == len(set(sums)), max(A) < 2 * p * p)
def F(n):                                        # largest Sidon set in {1..n}, by backtracking on differences
    best = [0]
    def go(A, diffs, nxt):
        best[0] = max(best[0], len(A))
        for a in range(nxt, n + 1):
            nd = {a - b for b in A}
            if not nd & diffs and len(nd) == len(A): go(A + [a], diffs | nd, a + 1)
    go([], set(), 1); return best[0]
print([(n, F(n), round(math.sqrt(n), 2)) for n in (10, 20, 30, 40)])

**Problem 5.20 (random polynomials).** Squarefree counts, the error in the irreducible count, and factor types for large $q$.

In [ ]:
x = sp.symbols('x')
for q in (2, 3):
    for n in (2, 3, 4):
        sf = sum(1 for co in itertools.product(range(q), repeat=n)
                 if sp.gcd(sp.Poly([1, *co], x, modulus=q), sp.Poly([1, *co], x, modulus=q).diff(x)).degree() == 0)
        print(q, n, sf, q**n - q**(n - 1))
print(all(abs(Fr(M(n, q), q**n) - Fr(1, n)) <= Fr(2, n) * q**Fr(-n, 2) for q in (2, 3, 4, 5, 7) for n in range(1, 16)))
def type_count(lam, q): return math.prod(comb(M(j, q) + m - 1, m) for j, m in lam.items())
for lam in sp.utilities.iterables.partitions(4):
    lam = dict(lam)
    print(lam, round(type_count(lam, 10007) / 10007**4, 5), float(Fr(1, math.prod(j**m * math.factorial(m) for j, m in lam.items()))))

**Problem 5.21 (roots).** The exact law of the number of roots, for $n\ge q$ and for $n<q$.

In [ ]:
def root_law(q, n):
    R = Counter()
    for co in itertools.product(range(q), repeat=n):
        R[sum(1 for a in range(q) if (a**n + sum(c * a**(n - 1 - i) for i, c in enumerate(co))) % q == 0)] += 1
    return {k: Fr(v, q**n) for k, v in sorted(R.items())}
for q, n in ((2, 2), (2, 3), (3, 3), (3, 4)):
    print(q, n, root_law(q, n) == {k: Fr(comb(q, k) * (q - 1)**(q - k), q**q) for k in range(q + 1) if comb(q, k) * (q - 1)**(q - k)})
for q, n in ((5, 2), (5, 3), (7, 3)):
    print(q, n, root_law(q, n)[0], sum(Fr((-1)**k * comb(q, k), q**k) for k in range(n + 1)))

**Problem 5.22 (two $k$-th powers).** The bound for all $p<400$ and $k\mid p-1$; the primes where some residue is missed.

In [ ]:
for p in primerange(5, 400):
    for k in divisors(p - 1):
        if k < 2: continue
        Nc = Counter((pow(a, k, p) + pow(b, k, p)) % p for a in range(p) for b in range(p))
        assert max(abs(Nc[c] - p) for c in range(1, p)) <= (k - 1) * (k - 2) * math.sqrt(p) + k - 1
        if p >= k**4: assert len(Nc) == p
fails = [(p, k) for p in primerange(5, 800) for k in divisors(p - 1) if k >= 2
         and len({(a + b) % p for a in {pow(t, k, p) for t in range(p)} for b in {pow(t, k, p) for t in range(p)}}) < p]
largest = {}
for p, k in fails: largest[k] = max(largest.get(k, 0), p)
print("bound ok; largest failing prime below 800 for each k:", sorted(largest.items())[:8])

## Part B: laboratory

**L5.1 Hunting caps.** Write a randomised greedy search for caps in $\mathbb F_3^n$: shuffle the points, add a point whenever it
does not complete a line with two chosen points. Run it many times for $n=2,3,4,5$ and compare the best cap found with the true
maxima $4,9,20,45$ and with the bound $3M_n$. How often does the greedy search find a maximum cap?

**L5.2 Two kinds of randomness in $a_p$.** For $x^3+y^3=1$ the normalised errors $a_p/(2\sqrt p)$ of Problem 5.3 seem to pile up
at $\pm1$. Plot their histogram for $p<20000$ against the arcsine density $\frac1{\pi\sqrt{1-x^2}}$. Then do the same for the
curve $y^2=x^3-x+1$, with $a_p=p-\#\{(x,y):y^2=x^3-x+1\}$, against the semicircle density $\frac2\pi\sqrt{1-x^2}$ (Sato--Tate).

**L5.3 Cliques in Paley graphs.** Compute the clique number of $P_p$ for all primes $p\equiv1\pmod4$ below $400$ and plot it
with $\sqrt p$ and $\log_2p$. Which bound looks closer to the truth?

**L5.4 Breaking an m-sequence.** Implement the Berlekamp--Massey algorithm over $\mathbb F_2$ and check that $2n$ bits of an
$m$-sequence of degree $n$ reveal its recurrence, for $x^5+x^2+1$ and for the GPS polynomial $x^{10}+x^3+1$. Then feed it random
bits: how long is the shortest recurrence that produces $N$ random bits?

## Solutions to the laboratory

### L5.1

In [ ]:
def third(a, b): return tuple((-x - y) % 3 for x, y in zip(a, b))
def greedy_cap(n, rng):
    pts = list(itertools.product(range(3), repeat=n)); rng.shuffle(pts)
    S, bad = [], set()
    for p in pts:
        if p in bad: continue
        for s in S: bad.add(third(p, s))
        S.append(p); bad.add(p)
    return len(S)
rng = random.Random(1)
for n, true_max, runs in ((2, 4, 200), (3, 9, 200), (4, 20, 400), (5, 45, 300)):
    sizes = Counter(greedy_cap(n, rng) for _ in range(runs))
    print(n, "best", max(sizes), "true max", true_max, "bound", 3 * Mn(n), " frequency of the best:", sizes[max(sizes)] / runs)

For $n\le3$ greedy finds the maximum often. For $n=4$ it stops at $18$ or below in every run, never reaching $20$, and for
$n=5$ it stalls near $38$, well below $45$: a maximal cap (one that cannot be extended) need not be maximum, and the large caps are very structured. The
bound $3M_n$ is far above the truth for these small $n$; its strength is only asymptotic.

### L5.2

In [ ]:
P = [p for p in primerange(7, 20000) if p % 3 == 1]
cub = np.array([(cubic_count(p) - p + 2) / (2 * math.sqrt(p)) for p in P])
def ec_ap(p):                         # a_p for y^2 = x^3 - x + 1
    sq = Counter(y * y % p for y in range(p))
    return p - sum(sq[(x**3 - x + 1) % p] for x in range(p))
Q = [p for p in primerange(5, 20000) if (4 * (-1)**3 + 27) % p != 0]
ell = np.array([ec_ap(p) / (2 * math.sqrt(p)) for p in Q])
t = np.linspace(-0.999, 0.999, 400)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(cub, bins=30, density=True, alpha=0.6); ax[0].plot(t, 1 / (np.pi * np.sqrt(1 - t**2))); ax[0].set_title("x^3+y^3=1 (CM): arcsine")
ax[1].hist(ell, bins=30, density=True, alpha=0.6); ax[1].plot(t, 2 / np.pi * np.sqrt(1 - t**2)); ax[1].set_title("y^2=x^3-x+1: semicircle")
plt.show()
print("second moments:", round(float(np.mean(cub**2)), 3), "(arcsine 1/2)", round(float(np.mean(ell**2)), 3), "(semicircle 1/4)")

The cubic Fermat curve has complex multiplication: $a_p=2\operatorname{Re}\pi$ with $\pi\in\Z[\omega]$ of norm $p$, and the angle of
$\pi$ is equidistributed (Hecke), so $a_p/(2\sqrt p)=\cos\theta$ with $\theta$ uniform: the arcsine law, mean square $\frac12$.
The second curve has no complex multiplication, and its angles follow the Sato--Tate law $\frac2\pi\sin^2\theta\,d\theta$, proved
by Clozel, Harris, Shepherd-Barron and Taylor (2008): mean square $\frac14$.

### L5.3

In [ ]:
Ps = [p for p in primerange(5, 400) if p % 4 == 1]
om = [paley_clique(p) for p in Ps]
plt.figure(figsize=(6, 3.5))
plt.plot(Ps, om, 'o', ms=3, label="clique number")
plt.plot(Ps, np.sqrt(Ps), label="sqrt p"); plt.plot(Ps, np.log2(Ps), label="log2 p"); plt.legend(); plt.show()
print(list(zip(Ps, om))[-6:])

The clique numbers stay close to $\log_2p$, far below $\sqrt p$. The truth is believed to be $O(\log^2p)$ or even $O(\log p\cdot
\log\log p)$; it is known that the clique number is at least about $\log p$ infinitely often (Graham and Ringrose), and nothing
much better than $\sqrt{p/2}$ is known as an upper bound.

### L5.4

In [ ]:
def berlekamp_massey(s):              # shortest LFSR over F_2: returns (L, C) with sum_{i<=L} C[i] s[k-i] = 0
    C, B, L, m = [1], [1], 0, 1
    for k in range(len(s)):
        d = s[k]
        for i in range(1, L + 1): d ^= C[i] & s[k - i]
        if d == 0: m += 1; continue
        T = C[:]; C = C + [0] * (len(B) + m - len(C))
        for i, b in enumerate(B): C[i + m] ^= b
        if 2 * L <= k: L, B, m = k + 1 - L, T, 1
        else: m += 1
    return L, C[:L + 1]
for taps, n in (([0, 2], 5), ([0, 3], 10)):
    s = lfsr(taps, n, 4 * n, [1] + [0] * (n - 1))
    print(n, berlekamp_massey(s[:2 * n]), " confirmed on 4n bits:", berlekamp_massey(s)[0] == n)
rng = random.Random(2)
for N in (20, 100, 400):
    print(N, "random bits need a recurrence of length", berlekamp_massey([rng.randrange(2) for _ in range(N)])[0])

The connection polynomial $C$ returned for $x^5+x^2+1$ is $1+x^3+x^5$, the reciprocal polynomial, since it reads the recurrence
backwards: $s_k=s_{k-3}+s_{k-5}$ is the same as $s_{k+5}=s_{k+2}+s_k$. Ten bits break the degree-$5$ generator and twenty the
degree-$10$ one. Random bits, by contrast, need a recurrence of length about $\frac N2$: the \emph{linear complexity} of a random
sequence grows like half its length, which is the property a cryptographic stream must have and an $m$-sequence lacks.